In [1]:
import os
import shutil
from glob import glob

root_data_dir = './data'
output_base = './class_split'  # 클래스별 결과 저장 폴더
os.makedirs(output_base, exist_ok=True)

country_dirs = [d for d in os.listdir(root_data_dir) if d.startswith('country_')]

# 클래스 수만큼 반복
for cls_id in range(4):
    img_out = os.path.join(output_base, f'class_{cls_id}/images')
    lbl_out = os.path.join(output_base, f'class_{cls_id}/labels')
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    # 각 country 디렉토리마다 반복
    for country in country_dirs:
        original_img_dir = os.path.join(root_data_dir, country, 'images')
        original_lbl_dir = os.path.join(root_data_dir, country, 'labels')

        label_files = sorted(glob(f"{original_lbl_dir}/*.txt"))

        for lbl_path in label_files:
            with open(lbl_path, 'r') as f:
                lines = f.readlines()

            # 해당 클래스에 해당하는 라벨만 추출
            filtered = []
            for line in lines:
                if line.strip().startswith(str(cls_id) + ' '):
                    # 클래스 ID를 0으로 변경 (나머지 좌표는 그대로 유지)
                    parts = line.strip().split()
                    parts[0] = '0'  # 클래스 ID를 0으로 변경
                    filtered.append(' '.join(parts) + '\n')

            if filtered:
                # 이미지 파일도 같이 복사
                base = os.path.basename(lbl_path).replace('.txt', '')
                for ext in ['.jpg', '.png']:
                    img_path = os.path.join(original_img_dir, base + ext)
                    if os.path.exists(img_path):
                        shutil.copy(img_path, os.path.join(img_out, base + ext))
                        break

                with open(os.path.join(lbl_out, base + '.txt'), 'w') as f:
                    f.writelines(filtered)

print("✅ 모든 country_*에 대해 클래스별 분할 완료!")

✅ 모든 country_*에 대해 클래스별 분할 완료!


In [7]:
# 📦 필요 모듈
import os
import shutil
import random
from glob import glob

# 📂 데이터셋 분할 함수
def split_dataset(base_dir, class_id, train_ratio=0.7, val_ratio=0.15):
    img_dir = os.path.join(base_dir, f'class_{class_id}/images')
    lbl_dir = os.path.join(base_dir, f'class_{class_id}/labels')

    img_paths = sorted(glob(f'{img_dir}/*.jpg') + glob(f'{img_dir}/*.png'))
    random.shuffle(img_paths)

    n = len(img_paths)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    splits = {
        'train': img_paths[:train_end],
        'val': img_paths[train_end:val_end],
        'test': img_paths[val_end:]
    }

    for split in splits:
        os.makedirs(f'{base_dir}/class_{class_id}/images/{split}', exist_ok=True)
        os.makedirs(f'{base_dir}/class_{class_id}/labels/{split}', exist_ok=True)

    for split, paths in splits.items():
        for img_path in paths:
            base = os.path.basename(img_path).rsplit('.', 1)[0]
            lbl_path = os.path.join(lbl_dir, base + '.txt')

            shutil.copy(img_path, f'{base_dir}/class_{class_id}/images/{split}/')
            shutil.copy(lbl_path, f'{base_dir}/class_{class_id}/labels/{split}/')

    print(f"📂 class_{class_id} → train/val/test 분할 완료 ({n}개)")

# ✅ 실제 실행
for cls_id in range(4):
    split_dataset('./class_split', cls_id)


📂 class_0 → train/val/test 분할 완료 (1817개)
📂 class_1 → train/val/test 분할 완료 (2934개)
📂 class_1 → train/val/test 분할 완료 (2934개)
📂 class_2 → train/val/test 분할 완료 (2723개)
📂 class_2 → train/val/test 분할 완료 (2723개)
📂 class_3 → train/val/test 분할 완료 (3082개)
📂 class_3 → train/val/test 분할 완료 (3082개)


In [6]:
import yaml
import os

def create_data_yaml(base_dir, class_id, class_name):
    # Get absolute path to avoid path resolution issues
    abs_base_dir = os.path.abspath(base_dir)
    
    data = {
        'train': f'{abs_base_dir}/class_{class_id}/images/train',
        'val': f'{abs_base_dir}/class_{class_id}/images/val',
        'test': f'{abs_base_dir}/class_{class_id}/images/test',
        'nc': 1,
        'names': [class_name]
    }

    yaml_path = os.path.join(base_dir, f'class_{class_id}/data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)

    print(f"📄 class_{class_id} → data.yaml 생성 완료")

# 클래스 이름이 있는 경우
class_names = ['Pothole', 'Alligator Crack', 'Transverse Crack', 'Longitudinal Crack']
for cls in range(4):
    create_data_yaml('./class_split', cls, class_names[cls])

📄 class_0 → data.yaml 생성 완료
📄 class_1 → data.yaml 생성 완료
📄 class_2 → data.yaml 생성 완료
📄 class_3 → data.yaml 생성 완료


In [ ]:
import os
from ultralytics import YOLO

# for cls in range(4):
#     model = YOLO('yolov8m.pt')
#     model.train(
#         data=f'class_split/class_{cls}/data.yaml',
#         epochs=100,
#         imgsz=640,
#         batch=16,
#         project='cls_yolo_runs',
#         name=f'class_{cls}'
#     )

model = YOLO('yolov8m.pt')
model.train(
    data=f'class_split/class_1/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    project='cls_yolo_runs',
    name=f'class_1'
)

Ultralytics 8.3.169 🚀 Python-3.12.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=class_split/class_0/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=class_07, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, p

train: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/class_split/class_0/labels/train... 1643 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1643/1643 [00:01<00:00, 1074.00it/s]
train: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/class_split/class_0/labels/train... 1643 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1643/1643 [00:01<00:00, 1074.00it/s]


train: New cache created: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/class_split/class_0/labels/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2177.6±1074.9 MB/s, size: 116.1 KB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2177.6±1074.9 MB/s, size: 116.1 KB)


val: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/class_split/class_0/labels/val... 503 images, 0 backgrounds, 0 corrupt: 100%|██████████| 503/503 [00:00<00:00, 1086.54it/s]

val: New cache created: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/class_split/class_0/labels/val.cache


Plotting labels to cls_yolo_runs/class_07/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to cls_yolo_runs/class_07
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to cls_yolo_runs/class_07
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss  

      1/100      6.24G      2.303      2.988      1.886         23        640: 100%|██████████| 103/103 [00:20<00:00,  5.05it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):   0%|          | 0/16 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:03<00:00,  5.29it/s]



                   all        503        884     0.0528       0.31     0.0376     0.0117

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      2/100      7.44G      2.368      2.509      1.964         31        640:  88%|████████▊ | 91/103 [00:17<00:02,  5.34it/s]